# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR<sup>2</sup> dataset defined by a [Croissant schema](https://mlcommons.org/croissant/). We'll use the `mlcroissant` library for programmatic access to both dataset metadata and records.

### Dataset Source
The dataset source is provided via its Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata is an object. For printing, use properties directly.
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. These IDs determine how we access and process data from each record set using `mlcroissant`.

**Note:** All record sets, fields, and columns are referenced by their `@id` for consistency.

In [ ]:
# List available RecordSet @ids and their fields
print("Available record sets and their fields (referenced by @id):\n")

record_sets = dataset.metadata.record_sets
for rset in record_sets:
    print(f"RecordSet @id: {rset.id}")
    print(f"  Name: {rset.name}")
    print(f"  Description: {getattr(rset, 'description', '')}")
    print("  Field @ids:")
    if hasattr(rset, 'fields'):
        for field in rset.fields:
            print(f"    - {field.id}: {field.name}")
    print("\n")
# For demonstration, display the first record for each record set
for rset in record_sets:
    print(f"First record from RecordSet @id={rset.id}:")
    records = list(dataset.records(record_set=rset.id))
    if records:
        print(json.dumps(records[0], indent=2))
    else:
        print("  No records found.")
    print("\n")

## 3. Data Extraction
We'll extract all records from each record set by their `@id` into Pandas DataFrames for further analysis. Use the record set and field `@id`s discovered above.

Modify the list of `record_sets_ids` according to your dataset contents as needed.

In [ ]:
# Collect all record set @ids (list of strings)
record_sets_ids = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from RecordSet @id={record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} records with columns: {df.columns.tolist()}")
    else:
        print("  No records found.")
    print()
# For demonstration: show the first 5 rows of the first available DataFrame
if dataframes:
    first_rsid = next(iter(dataframes.keys()))
    print(f"First rows of DataFrame for RecordSet @id={first_rsid}")
    display(dataframes[first_rsid].head())

## 4. Exploratory Data Analysis (EDA)
We'll process a numeric field in one of the main record sets. Typical steps include filtering records, normalizing a numeric field, and grouping by a categorical field.

Set the following variables appropriately using the `@id`s discovered above:
- `main_record_set_id`: the record set to use
- `numeric_field_id`: a numeric field (e.g. peptide count, molecular weight, etc.)
- `group_field_id`: a categorical field to group by (e.g., description, accession, etc.)

In [ ]:
# Example: Pick main record set and fields by their @id
# List all loaded dataframes and their columns
print('Available DataFrames and their fields:')
for rsid, df in dataframes.items():
    print(f'RecordSet @id: {rsid}')
    print(f'  Columns: {df.columns.tolist()}')
    print()

# Set variables for EDA -- you may need to replace these with the correct @id values for your dataset
main_record_set_id = list(dataframes.keys())[0]  # Use the first record set loaded
df = dataframes[main_record_set_id]
# Attempt to find a numeric field, e.g., a peptide count, abundance, or MW. If unsure, pick one from columns.
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'if' and not df[col].isnull().all()]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback

# Try to group by a different field
possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field_id = possible_group_fields[0] if possible_group_fields else None

print(f'Using RecordSet @id: {main_record_set_id}')
print(f'Numeric field: {numeric_field_id}')
print(f'Group field: {group_field_id}')

# Filtering, normalizing, and grouping
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={filtered_df.shape[0]})")
    display(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group if field available
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f'Mean_{numeric_field_id}').reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the numeric field's distribution and its relation to the group/categorical field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30)
plt.title(f'Distribution of {numeric_field_id} in RecordSet @id={main_record_set_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group field, boxplot of numeric field by group
if group_field_id is not None:
    plt.figure(figsize=(10, 5))
    order = df[group_field_id].value_counts().index[:10]  # Show top 10 groups
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(order)], order=order)
    plt.title(f'{numeric_field_id} by {group_field_id} (top 10)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR<sup>2</sup> Croissant dataset using the `mlcroissant` library:
- Loaded dataset metadata and discovered record sets and their fields using each entity's `@id`.
- Extracted all records and loaded them into Pandas DataFrames for flexible analysis.
- Performed exploratory data analysis on a numeric field with filtering, normalization, grouping, and visualizations.

**Next steps:** Extend this notebook by integrating downstream ML tasks, cross-referencing record sets, or detailed statistical analyses based on the available field semantics and domain needs.